In [77]:
import pandas as pd
import numpy as np

In [78]:
kdd_train= pd.read_csv("kdd_train.csv")

In [79]:
kdd_test= pd.read_csv("kdd_test.csv")

In [80]:
kdd_train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,last_flag,label_5,label_2
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,Normal,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,Normal,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,DoS,attack
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,Normal,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,Normal,normal


In [81]:
kdd_test.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack,last_flag,label_5,label_2
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21,DoS,attack
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21,DoS,attack
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.02,0.0,0.0,0.00,0.00,normal,21,Normal,normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.28,0.0,0.0,0.00,0.00,saint,15,Probe,attack
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11,Probe,attack


In [82]:
print("kdd_train:",kdd_train.shape)

kdd_train: (125973, 45)


In [83]:
print("kdd_test:",kdd_test.shape)

kdd_test: (22544, 45)


#### Importing categorical columns list

In [84]:
categ_content=['hot','num_failed_logins','num_file_creations','num_shells','num_access_files']
categ_basic=['protocol_type','service','flag','wrong_fragment','urgent','last_flag']

categ_columns=categ_basic + categ_content


In [85]:
kdd_train[categ_columns].head()

,protocol_type,service,flag,wrong_fragment,urgent,last_flag,hot,num_failed_logins,num_file_creations,num_shells,num_access_files
0,tcp,ftp_data,SF,0,0,20,0,0,0,0,0
1,udp,other,SF,0,0,15,0,0,0,0,0
2,tcp,private,S0,0,0,19,0,0,0,0,0
3,tcp,http,SF,0,0,21,0,0,0,0,0
4,tcp,http,SF,0,0,21,0,0,0,0,0


In [86]:
kdd_test[categ_columns].head()

,protocol_type,service,flag,wrong_fragment,urgent,last_flag,hot,num_failed_logins,num_file_creations,num_shells,num_access_files
0,tcp,private,REJ,0,0,21,0,0,0,0,0
1,tcp,private,REJ,0,0,21,0,0,0,0,0
2,tcp,ftp_data,SF,0,0,21,0,0,0,0,0
3,icmp,eco_i,SF,0,0,15,0,0,0,0,0
4,tcp,telnet,RSTO,0,0,11,0,0,0,0,0


In [87]:
##Saving labels to a dataframe for reference
label_list=[ 'attack','label_2','label_5']
kdd_train_label=kdd_train[label_list]
print(kdd_train_label.head())
print("kdd_train_label:", kdd_train_label.shape)

    attack label_2 label_5
0   normal  normal  Normal
1   normal  normal  Normal
2  neptune  attack     DoS
3   normal  normal  Normal
4   normal  normal  Normal
kdd_train_label: (125973, 3)


In [88]:
kdd_test_label=kdd_test[label_list]
print(kdd_test_label.head())
print("kdd_test_label:", kdd_test_label.shape)

    attack label_2 label_5
0  neptune  attack     DoS
1  neptune  attack     DoS
2   normal  normal  Normal
3    saint  attack   Probe
4    mscan  attack   Probe
kdd_test_label: (22544, 3)


In [89]:
#su_attempted takes only values 0 and 1
kdd_train['su_attempted'].replace(2,0,inplace=True)
#su_attempted takes only values 0 and 1
kdd_test['su_attempted'].replace(2,0,inplace=True)
#num_file_creations shows an abnormal value 100.Repalcing that with
#majority value 0
kdd_test['num_file_creations'].replace(100,0,inplace=True)

In [90]:
insig_list=['num_outbound_cmds','is_host_login']
corr_drop_list=['num_root','srv_serror_rate','srv_rerror_rate','dst_host_srv_serror_rate','dst_host_srv_rerror_rate',
             'dst_host_serror_rate','dst_host_rerror_rate','dst_host_same_srv_rate']


In [91]:
##Columns to be dropped after Encoding
to_drop_list= label_list +insig_list + corr_drop_list
print(to_drop_list)

['attack', 'label_2', 'label_5', 'num_outbound_cmds', 'is_host_login', 'num_root', 'srv_serror_rate', 'srv_rerror_rate', 'dst_host_srv_serror_rate', 'dst_host_srv_rerror_rate', 'dst_host_serror_rate', 'dst_host_rerror_rate', 'dst_host_same_srv_rate']


In [92]:
kdd_train.drop(to_drop_list,axis=1,inplace=True)
kdd_train.shape

(125973, 32)

In [93]:

kdd_test.drop(to_drop_list,axis=1,inplace=True)
kdd_test.shape

(22544, 32)

### ONE HOT ENCODING

from sklearn.preprocessing import OneHotEncoder


enc = OneHotEncoder(handle_unknown='ignore')

train_enc = pd.DataFrame(enc.fit_transform(kdd_train[categ_columns]).toarray())
kdd_train_enc = kdd_train.join(train_enc)
kdd_train_enc.head()

test_enc = pd.DataFrame(enc.fit_transform(kdd_test[categ_columns]).toarray())
kdd_test_enc = kdd_test.join(train_enc)
kdd_test_enc.head()

kdd_train_enc.drop(to_drop_list, axis = 1, inplace = True)

kdd_test_enc.drop(to_drop_list, axis = 1, inplace = True)

kdd_train_enc.describe()

kdd_test_enc.describe()

### Dummy variable Encoding

In [94]:
#Selecting columns to create dummy variables
dummy_Var_list= categ_columns
print(dummy_Var_list)

['protocol_type', 'service', 'flag', 'wrong_fragment', 'urgent', 'last_flag', 'hot', 'num_failed_logins', 'num_file_creations', 'num_shells', 'num_access_files']


In [95]:
# Taking the data for categorical variables into a dictionary 

Dict_values={}
for col_name in dummy_Var_list:

        Dict_values[col_name]=kdd_train[col_name].unique()
    
string=""    
    
for key, value in Dict_values.items():
    string+=str(key) + ":" +str(value) + "\n"

    
print(string)


protocol_type:['tcp' 'udp' 'icmp']
service:['ftp_data' 'other' 'private' 'http' 'remote_job' 'name' 'netbios_ns'
 'eco_i' 'mtp' 'telnet' 'finger' 'domain_u' 'supdup' 'uucp_path' 'Z39_50'
 'smtp' 'csnet_ns' 'uucp' 'netbios_dgm' 'urp_i' 'auth' 'domain' 'ftp'
 'bgp' 'ldap' 'ecr_i' 'gopher' 'vmnet' 'systat' 'http_443' 'efs' 'whois'
 'imap4' 'iso_tsap' 'echo' 'klogin' 'link' 'sunrpc' 'login' 'kshell'
 'sql_net' 'time' 'hostnames' 'exec' 'ntp_u' 'discard' 'nntp' 'courier'
 'ctf' 'ssh' 'daytime' 'shell' 'netstat' 'pop_3' 'nnsp' 'IRC' 'pop_2'
 'printer' 'tim_i' 'pm_dump' 'red_i' 'netbios_ssn' 'rje' 'X11' 'urh_i'
 'http_8001' 'aol' 'http_2784' 'tftp_u' 'harvest']
flag:['SF' 'S0' 'REJ' 'RSTR' 'SH' 'RSTO' 'S1' 'RSTOS0' 'S3' 'S2' 'OTH']
wrong_fragment:[0 3 1]
urgent:[0 1 3 2]
last_flag:[20 15 19 21 18 17 16 12 14 11  2 13 10  9  8  7  3  5  1  6  0  4]
hot:[ 0  5  6  4  2  1 28 30 22 24 14  3 15 25 19 18 77 17 11  7 20 12  9 10
  8 21 33 44]
num_failed_logins:[0 2 1 3 4 5]
num_file_creations:[ 0  

#### Function to create dummy variable

In [96]:
#Function to craete dummy variables
def dummy_variable(col_name,data):
    #Create Dummies for the variable
    
    temp_dm=pd.get_dummies(data[col_name], drop_first = False)
    
    #Prefixing the dummy columns with appropriate name
    pref_col=str(col_name)+ "_"
    temp_dm=temp_dm.add_prefix(pref_col)
    
    return temp_dm

#### Adding dummy variables dropping one level to the Train dataset

In [97]:
data= kdd_train.copy()
dummy_data= pd.DataFrame()
for col in dummy_Var_list:
    temp_data=dummy_variable(col,data)
       
    #Conacatenating the dummy variables to original dataset
    dummy_data = pd.concat([dummy_data,temp_data], axis = 1)
    
    
dummy_data.head()

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_aol,service_auth,service_bgp,service_courier,...,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4,num_access_files_5,num_access_files_6,num_access_files_7,num_access_files_8,num_access_files_9
0,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [98]:
kdd_train_dm = pd.concat([kdd_train,dummy_data], axis = 1)
     
kdd_train_dm.drop(dummy_Var_list, axis = 1, inplace = True)
        
print("Dummy variables added")

Dummy variables added


In [99]:
kdd_train_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Columns: 216 entries, duration to num_access_files_9
dtypes: float64(8), int64(13), uint8(195)
memory usage: 43.6 MB


In [100]:
kdd_train_dm.shape

(125973, 216)

In [101]:
kdd_train_dm.describe()

,duration,src_bytes,dst_bytes,land,logged_in,num_compromised,root_shell,su_attempted,is_guest_login,count,...,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4,num_access_files_5,num_access_files_6,num_access_files_7,num_access_files_8,num_access_files_9
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.00000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.395736,0.279250,0.001342,0.000167,0.009423,84.107555,...,0.997055,0.002485,0.000230,0.000064,0.00004,0.000048,0.000032,0.000016,0.000024,0.000008
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.489010,23.942042,0.036603,0.012910,0.096612,114.508607,...,0.054189,0.049785,0.015171,0.007969,0.00630,0.006901,0.005635,0.003985,0.004880,0.002817
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,...,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,...,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,143.000000,...,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,1.000000,7479.000000,1.000000,1.000000,1.000000,511.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000


#### Adding dummy variables dropping one level to the Test dataset

In [102]:
data= kdd_test.copy()
dummy_data= pd.DataFrame()
for col in dummy_Var_list:
    temp_data=dummy_variable(col,data)
       
    #Conacatenating the dummy variables to original dataset
    dummy_data = pd.concat([dummy_data,temp_data], axis = 1)
    
    
dummy_data.head()

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_auth,service_bgp,service_courier,service_csnet_ns,...,num_file_creations_7,num_shells_0,num_shells_1,num_shells_2,num_shells_5,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4
0,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0


In [103]:
kdd_test_dm = pd.concat([kdd_test,dummy_data], axis = 1)
     
kdd_test_dm.drop(dummy_Var_list, axis = 1, inplace = True)
        
print("Dummy variables added")

Dummy variables added


In [104]:
kdd_test_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Columns: 166 entries, duration to num_access_files_4
dtypes: float64(8), int64(13), uint8(145)
memory usage: 6.7 MB


In [105]:
kdd_test_dm.shape

(22544, 166)

In [106]:
kdd_test_dm.describe()

,duration,src_bytes,dst_bytes,land,logged_in,num_compromised,root_shell,su_attempted,is_guest_login,count,...,num_file_creations_7,num_shells_0,num_shells_1,num_shells_2,num_shells_5,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4
count,22544.000000,2.254400e+04,2.254400e+04,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,...,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000
mean,218.859076,1.039545e+04,2.056019e+03,0.000311,0.442202,0.119899,0.002440,0.000089,0.028433,79.028345,...,0.000044,0.999157,0.000665,0.000133,0.000044,0.996806,0.002972,0.000133,0.000044,0.000044
std,1407.176612,4.727864e+05,2.121930e+04,0.017619,0.496659,7.269597,0.049334,0.009419,0.166211,128.539248,...,0.006660,0.029019,0.025787,0.011535,0.006660,0.056424,0.054436,0.011535,0.006660,0.006660
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,5.400000e+01,4.600000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,2.870000e+02,6.010000e+02,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,123.250000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
max,57715.000000,6.282565e+07,1.345927e+06,1.000000,1.000000,796.000000,1.000000,1.000000,1.000000,511.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Normalization


## Minmax scaling of Train and test dataset

In [107]:
kdd_train_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Columns: 216 entries, duration to num_access_files_9
dtypes: float64(8), int64(13), uint8(195)
memory usage: 43.6 MB


In [108]:
kdd_test_dm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Columns: 166 entries, duration to num_access_files_4
dtypes: float64(8), int64(13), uint8(145)
memory usage: 6.7 MB


In [109]:
col_list_train=kdd_train_dm.columns

In [110]:
col_list_test=kdd_test_dm.columns

In [111]:
#Importing Minmax scaler 
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [112]:
kdd_train_norm=scaler.fit_transform(kdd_train_dm)
kdd_train_norm= pd.DataFrame(kdd_train_norm, columns= col_list_train)

In [113]:
kdd_test_norm=scaler.fit_transform(kdd_test_dm)
kdd_test_norm= pd.DataFrame(kdd_test_norm, columns= col_list_test)

In [114]:
kdd_train_norm.describe()

,duration,src_bytes,dst_bytes,land,logged_in,num_compromised,root_shell,su_attempted,is_guest_login,count,...,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4,num_access_files_5,num_access_files_6,num_access_files_7,num_access_files_8,num_access_files_9
count,125973.000000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.00000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,0.006692,3.302024e-05,1.509928e-05,0.000198,0.395736,0.000037,0.001342,0.000167,0.009423,0.164594,...,0.997055,0.002485,0.000230,0.000064,0.00004,0.000048,0.000032,0.000016,0.000024,0.000008
std,0.060700,4.253974e-03,3.069818e-03,0.014086,0.489010,0.003201,0.036603,0.012910,0.096612,0.224087,...,0.054189,0.049785,0.015171,0.007969,0.00630,0.006901,0.005635,0.003985,0.004880,0.002817
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003914,...,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,3.188489e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.027397,...,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,2.000052e-07,3.939120e-07,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.279843,...,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000e+00,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000


In [115]:
kdd_test_norm.describe()

,duration,src_bytes,dst_bytes,land,logged_in,num_compromised,root_shell,su_attempted,is_guest_login,count,...,num_file_creations_7,num_shells_0,num_shells_1,num_shells_2,num_shells_5,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4
count,22544.000000,2.254400e+04,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,...,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000
mean,0.003792,1.654651e-04,0.001528,0.000311,0.442202,0.000151,0.002440,0.000089,0.028433,0.154654,...,0.000044,0.999157,0.000665,0.000133,0.000044,0.996806,0.002972,0.000133,0.000044,0.000044
std,0.024381,7.525373e-03,0.015766,0.017619,0.496659,0.009133,0.049334,0.009419,0.166211,0.251545,...,0.006660,0.029019,0.025787,0.011535,0.006660,0.056424,0.054436,0.011535,0.006660,0.006660
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001957,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,8.595216e-07,0.000034,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.015656,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,4.568198e-06,0.000447,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.241194,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [116]:
kdd_train_norm = pd.concat([kdd_train_norm,kdd_train_label], axis = 1)
kdd_train_norm.head()

,duration,src_bytes,dst_bytes,land,logged_in,num_compromised,root_shell,su_attempted,is_guest_login,count,...,num_access_files_3,num_access_files_4,num_access_files_5,num_access_files_6,num_access_files_7,num_access_files_8,num_access_files_9,attack,label_2,label_5
0,0.0,3.558064e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.003914,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal,normal,Normal
1,0.0,1.057999e-07,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.025440,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal,normal,Normal
2,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.240705,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
3,0.0,1.681203e-07,6.223962e-06,0.0,1.0,0.0,0.0,0.0,0.0,0.009785,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal,normal,Normal
4,0.0,1.442067e-07,3.206260e-07,0.0,1.0,0.0,0.0,0.0,0.0,0.058708,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,normal,normal,Normal


In [117]:
kdd_test_norm = pd.concat([kdd_test_norm,kdd_test_label], axis = 1)
kdd_test_norm.head()

,duration,src_bytes,dst_bytes,land,logged_in,num_compromised,root_shell,su_attempted,is_guest_login,count,...,num_shells_2,num_shells_5,num_access_files_0,num_access_files_1,num_access_files_2,num_access_files_3,num_access_files_4,attack,label_2,label_5
0,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.448141,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
1,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.266145,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,neptune,attack,DoS
2,0.000035,2.066513e-04,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.001957,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,normal,normal,Normal
3,0.000000,3.183413e-07,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.001957,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,saint,attack,Probe
4,0.000017,0.000000e+00,0.000011,0.0,0.0,0.0,0.0,0.0,0.0,0.001957,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,mscan,attack,Probe


In [118]:
print("kdd_train_norm:",kdd_train_norm.shape)
print("kdd_test_norm:",kdd_test_norm.shape)

kdd_train_norm: (125973, 219)
kdd_test_norm: (22544, 169)


In [119]:
kdd_train_norm.to_csv("kdd_train_prep1.csv",index=False)

In [120]:
kdd_test_norm.to_csv("kdd_test_prep1.csv",index=False)